<img src = "../assets/bbai-banner.png" align="center" height = "25%">

# Schwan PdM Demonstration #

Each Machine is made up of 4 components.  In the original dataset, we were given 100 machine definitions that included:
 - Machine ID
 - Machine Model type
 - Machine Age
 
We were also given a large spreadsheet that had raw telemetry spanning one year for all 100 machines including:
 - Datetime
 - Machine ID
 - Vibration
 - Rotation
 - Voltage
 - Pressure

We were also given multiple other files that included errors, failures, and maintenance events that all had timestamps throughout the year.  From my best understanding the definitions are as follows
 - **Errors** : Minor problems with the machine.  Components were not attributed, but there were 5 types of errors present in the data.  We were given `datetime`, `machineID`, and `error` code
 - **Failures** : Major problems with the machine.  These were considered faults and were attributed to the component that failed.  We were given `datetime`, `machineID`, and `failure` indicating the component.
 - **Maintenance** : Logged maintenance events on components of the machines.  It was not indicated if the maintenance was scheduled or mandatory because of a **Failure**, but analysis indicated **Maintenance** events always followed **Failure** events for that component.  We were given `datetime`, `machineID`, and `comp` indicating the component serviced.  

 ### The objective of this notebook is to use the ZTDF PdM model to predict the remaining useful life of all four components for each machine.  ###

 We will use MLFlow to track models for each component (Labeled comp1, comp2, comp3, and comp4).  The data has been pre-processed into a format that will make it easier for use in the PdM RUL Prediction model using the notebook:
   - **Schwan_Data_Exploration.ipynb**

In [ ]:
# Take care of the Imports
import mlflow
import os
import sys
import pandas as pd
from mlflow.models import infer_signature

module_paths = [
    os.path.abspath(os.path.join('..')),
]
for module_path in module_paths:
    if module_path not in sys.path:
        sys.path.append(module_path)

from pdm.rul_model import RULModel
from pdm.config import RULConfig

In [ ]:
def open_data(file_path):
    # Identify the Column Names for the data
    column_names = [
        "datetime",
        "machineID",
        "voltage",
        "rotations",
        "pressure",
        "vibration",
        "machine_model",
        "machine_age",
        "cycle",
        "cyc_since_error1",
        "cyc_since_error2",
        "cyc_since_error3",
        "cyc_since_error4",
        "cyc_since_error5",
        "cyc_since_maint",  # Specific to the component.  Other maintenance not in the data (for now)
        "cyc_since_last_fail", # Specific to the component.  Other failure data not in the data (for now)
        "true_RUL",
        "unique_machine_id",
        ]

    # Load the file into Pandas dataframe
    train_data = pd.read_csv(file_path, sep=",", header=None, index_col=False, names=column_names)
    return train_data

In [ ]:
# Set up the sensor columns and load the data
sensor_cols = [
    "voltage",
    "rotations",
    "pressure",
    "vibration",
    # "machine_model",  # May need to strip down from model1 -> 1, model3 -> 3, etc.  
    # "machine_age",
    # "cyc_since_error1",  # May use a correlation matrix to see how much the errors affect the components.  
    # "cyc_since_error2",  # Might be able to use some of them but not others for different components.
    "cyc_since_error3",
    # "cyc_since_error4",
    "cyc_since_error5",
    "cyc_since_maint",  # This is specific to the component.  
    "cyc_since_last_fail",
]

# Set the tracking server uri for logging
# Don't forget to start the logging server!
mlflow.set_tracking_uri(uri="http://127.0.0.1:8080")

filename = "schwan_comp2"
filepath = '../data/schwan/'+filename+'.csv'
original_data = open_data(filepath)
original_data['datetime'] = pd.to_datetime(original_data['datetime'])
original_data['R_early'] = original_data['true_RUL']

original_data


In [ ]:
def prepare_training_data(training_data):
    columns_to_drop = [ 
        "datetime",
        "true_RUL",
        ]
    
    cleaned_data = training_data.copy()
    cleaned_data.drop(columns=columns_to_drop, inplace=True)
    # cleaned_data["cyc_since_error1"] = cleaned_data["cyc_since_error1"].apply(
    #         lambda x: 5000 if x is None else x
    #     )
    # cleaned_data["cyc_since_error2"] = cleaned_data["cyc_since_error2"].apply(
    #         lambda x: 5000 if x is None else x
    #     )
    # cleaned_data["cyc_since_error3"] = cleaned_data["cyc_since_error3"].apply(
    #         lambda x: 5000 if x is None else x
    #     )
    # cleaned_data["cyc_since_error4"] = cleaned_data["cyc_since_error4"].apply(
    #         lambda x: 5000 if x is None else x
    #     )
    # cleaned_data["cyc_since_error5"] = cleaned_data["cyc_since_error5"].apply(
    #         lambda x: 5000 if x is None else x
    #     )
        
    return cleaned_data

In [ ]:
training_data = prepare_training_data(original_data)
training_data.fillna(5000, inplace=True)
training_data.tail()

unique_id_counts = training_data['unique_machine_id'].value_counts()
print(unique_id_counts)

In [ ]:
save_file = "C:/models/"+filename+".h5"

# Create a new MLflow Experiment
mlflow.set_experiment("RUL Prediction Porting - "+filename)

mlflow.autolog(log_models=True, exclusive=True)
mlflow.tensorflow.autolog(log_models=True)

cfg = RULConfig(sensor_columns=sensor_cols, save_file_name=save_file, asset_id_column="unique_machine_id", cycle_column="cyc_since_last_fail", max_RUL=1200, seq_length=20)
model = RULModel(**cfg.model_dump())

with mlflow.start_run() as run:
    
    mlflow.log_params(cfg.model_dump())
    
    mlflow.set_tag("Notes", "Baseline - Component 2")
    model.batch_train(training_data)
mlflow.end_run()

### Idea Capture ###
Should I use the cyc_since_last_fail column as the cycle column, I will experiment with it on separate MLflow runs.  